In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /scratch/jieq/pandax/dias_notebooks/dataranch/supermarket-sales-prediction-xgboost-fastai/src/small_bench/checkpoints/post_cell_3.pickle

In [ ]:
%%cudf.pandas.profile
### cell 4 ###

### cell 4 optimized ###
# pull only numeric columns (already a GPU op)
numeric_df = df.select_dtypes(include=['number'])

# 1) compute column means (GPU)
mean_vals = numeric_df.mean()

# 2) center the data (GPU)
demeaned = numeric_df - mean_vals

# 3) form the covariance matrix on GPU
d_cov = (demeaned.T.dot(demeaned)) / (len(numeric_df) - 1)

# 4) get per-column standard deviations (GPU)
std_vals = numeric_df.std(ddof=1)

# 5) normalize covariance to get the correlation matrix (all on GPU)
#    divide columns by their std, then divide rows by their std
_ = d_cov.div(std_vals, axis=1).div(std_vals, axis=0)